# Typewell Attention v2 — pointer'ı gerçekten eğiten model

`sequence_based_with_typewell.ipynb`'in üzerine altı geliştirme. v1'de attention haritası uniform çıkmıştı: head görevi tek başına çözebildiği için attention'a gradient akmıyordu. v2 bunu hem öğrenme sinyali hem mimari tarafında düzeltir.

## Öğrenme sinyali

1. **Auxiliary pointer loss** — her tahmin satırının gerçek `dtvt`'si eğitimde bilinir (`target − anchor_scaled`). `soft_dtvt`'ye doğrudan MSE: attention "typewell'de doğru derinliği göster" sinyalini açıkça alır. `loss = task + λ_p · pointer + λ_s · smooth`.
2. **Pointer smoothness cezası** — TVT, MD boyunca süreklidir; işaret edilen derinlik timestep'ler arasında zıplayamaz. Ardışık `soft_dtvt` farklarının karesi cezalandırılır.

## Attention mekaniği

3. **Locality bias logit'lerde** — `−dtvt² / 2σ²` (σ öğrenilebilir) doğrudan pointer logit'ine eklenir: "anchor'a yakın derinlikler a priori daha olası" bilgisi mimariye gömülür.
4. **Ayrı, dropout'suz tek kafalı pointer attention** — context okuma eski 4 kafalı dropout'lu attention'da kalır; pointer için ayrı tek kafalı, dropout'suz, bias'lı bir attention keskin ve normalize bir dağılım üretir (renormalizasyon hilesine gerek kalmaz).

## Typewell encoder

5. **Conv1d encoder** — tek `Linear` yerine grid üzerinde dilation'ı artan 3 katmanlı Conv1d: her token çok ölçekli (≈±6/±12/±24 nokta) hiyerarşik bir GR imzası taşır. `multi_scale_ncc` fikrinin öğrenilebilir hali.
6. **Açık korelasyon kanalı** — her timestep'te horizontal GR'ın son 13 satırı ile her typewell token pattern'i arasındaki normalize çapraz korelasyon (NCC) hesaplanıp öğrenilebilir bir gain ile pointer logit'ine eklenir. PF/beam yöntemlerinin açık GR eşleşmesi, attention'a başlangıç haritası olarak verilir. (Not: MD-penceresi ile TVT-penceresi arasında eğime bağlı ölçek farkı vardır; bu bir prior'dır, gain işe yaramazsa modelce küçültülür.)

Veri pipeline'ı (split, feature'lar, typewell belleği, scaler'lar, episode kalibrasyonu, batch'leme) v1 ile birebir aynıdır. Checkpoint: `typewell_attn_v2_best.pt`.

In [1]:
import copy
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from training import (
    RNN_DROP_COLUMNS,
    RNN_FEATURE_COLUMNS,
    StatefulTVTWindowDataset,
    prepare_rnn_features,
)
from utils import (
    get_all_test_ids,
    get_all_training_ids,
    test_folder,
    train_folder,
)

SEED = 42
VALID_SIZE = 0.15
WINDOW_SIZE = 256
CONTEXT_LENGTH = 256
MIN_FORECAST_LENGTH = 256
MIN_FORECAST_FRACTION = 0.40
REAL_PS_ANCHOR_PROB = 0.70
REAL_PS_FRACTION_RANGE = (0.20, 0.35)
ANCHORS_PER_WELL = 8
EPISODE_BATCH_SIZE = 32
NUM_EPOCHS = 10

# v2 loss agirliklari (scaled TVT birimlerinde)
POINTER_LOSS_WEIGHT = 0.3     # soft_dtvt -> gercek dtvt MSE
SMOOTHNESS_LOSS_WEIGHT = 0.2  # ardisik soft_dtvt farklarinin karesi

CHECKPOINT_PATH = "typewell_attn_v2_best.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


/Users/doruksarigun/Kaggle/ROGII - Wellbore Geology Prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Veri pipeline'i — v1 (sequence_based_with_typewell.ipynb) ile birebir ayni.

all_training_ids = get_all_training_ids()
reserved_test_ids = set(get_all_test_ids())
modeling_ids = sorted(set(all_training_ids) - reserved_test_ids)

train_well_ids, valid_well_ids = train_test_split(
    modeling_ids,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_wells = prepare_rnn_features(
    training_ids=train_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
valid_wells = prepare_rnn_features(
    training_ids=valid_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
test_wells = prepare_rnn_features(
    training_ids=sorted(reserved_test_ids),
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
    data_dir=test_folder,
    target_data_dir=train_folder,
)
print(len(train_wells), len(valid_wells), len(test_wells))

654 116 3


In [3]:
# Typewell bellegi + scaler'lar (v1 ile ayni).

TYPEWELL_GRID_POINTS = 512
TYPEWELL_TOKEN_HALF_WINDOW = 6  # token = 13 komsu GR + dtvt


def build_typewell_grid(typewell_df, num_points=TYPEWELL_GRID_POINTS):
    typewell = (
        typewell_df.dropna(subset=["TVT", "GR"])
        .sort_values("TVT")
        .drop_duplicates("TVT")
    )
    if len(typewell) < 2:
        raise ValueError("Typewell icinde yeterli TVT-GR verisi yok.")
    tvt_grid = np.linspace(
        typewell["TVT"].min(), typewell["TVT"].max(), num_points
    ).astype(np.float32)
    gr_grid = np.interp(
        tvt_grid,
        typewell["TVT"].to_numpy(dtype=float),
        typewell["GR"].to_numpy(dtype=float),
    ).astype(np.float32)
    return tvt_grid, gr_grid


def fit_typewell_gr_calibration(known_tvt, known_gr, tvt_grid, gr_grid):
    in_range = (known_tvt >= tvt_grid[0]) & (known_tvt <= tvt_grid[-1])
    known_tvt = known_tvt[in_range]
    known_gr = known_gr[in_range]
    if len(known_tvt) < 3:
        return 1.0, 0.0
    expected_gr = np.interp(known_tvt, tvt_grid, gr_grid)
    scale, offset = np.polyfit(expected_gr, known_gr, 1)
    if not np.isfinite(scale) or abs(scale) < 1e-8:
        return 1.0, 0.0
    return float(scale), float(offset)


def known_calibration_pairs(well_data, tvt_column, gr_column="GR"):
    known = well_data.loc[
        well_data[tvt_column].notna() & (well_data["GR_isna"] == 0),
        [tvt_column, gr_column],
    ]
    return (
        known[tvt_column].to_numpy(dtype=float),
        known[gr_column].to_numpy(dtype=float),
    )


def build_typewell_gr_tokens(gr_grid, half_window=TYPEWELL_TOKEN_HALF_WINDOW):
    padded = np.pad(gr_grid, half_window, mode="edge")
    return np.lib.stride_tricks.sliding_window_view(
        padded, 2 * half_window + 1
    ).astype(np.float32)


def attach_typewell_memory(prepared_wells):
    for well in tqdm(prepared_wells.values(), desc="Typewell memory", unit="well"):
        tvt_grid, gr_grid = build_typewell_grid(well.typewell_df)
        known_tvt, known_gr = known_calibration_pairs(
            well.data, "observed_TVT"
        )
        scale, offset = fit_typewell_gr_calibration(
            known_tvt, known_gr, tvt_grid, gr_grid
        )
        well.tw_tvt_grid = tvt_grid
        well.tw_gr_grid = gr_grid
        well.tw_gr_tokens_raw = build_typewell_gr_tokens(gr_grid)
        well.tw_gr_tokens = well.tw_gr_tokens_raw * scale + offset
        well.tw_gr_calibration = (scale, offset)


attach_typewell_memory(train_wells)
attach_typewell_memory(valid_wells)
attach_typewell_memory(test_wells)

MASK_FEATURE_COLUMNS = [
    "GR_isna",
    "GR_typewell_imputed_mask",
    "GR_neighbor_imputed_mask",
]
CONTINUOUS_FEATURE_COLUMNS = [
    column for column in RNN_FEATURE_COLUMNS
    if column not in MASK_FEATURE_COLUMNS
]

feature_scaler = StandardScaler()
target_scaler = StandardScaler()
for well in train_wells.values():
    feature_scaler.partial_fit(well.data[CONTINUOUS_FEATURE_COLUMNS])
    target_scaler.partial_fit(well.data[["TVT"]])

target_mean = float(target_scaler.mean_[0])
target_scale = float(target_scaler.scale_[0])
_gr_channel = CONTINUOUS_FEATURE_COLUMNS.index("GR")
gr_mean = float(feature_scaler.mean_[_gr_channel])
gr_scale = float(feature_scaler.scale_[_gr_channel])

# Model iceride NCC icin horizontal GR kanalina ihtiyac duyar.
GR_FEATURE_INDEX = RNN_FEATURE_COLUMNS.index("GR")


def apply_scalers(prepared_wells):
    for well in prepared_wells.values():
        well.data.loc[:, "GR_raw"] = well.data["GR"]
        well.data.loc[:, CONTINUOUS_FEATURE_COLUMNS] = (
            feature_scaler.transform(well.data[CONTINUOUS_FEATURE_COLUMNS])
        )
        well.data.loc[:, "TVT_scaled"] = target_scaler.transform(
            well.data[["TVT"]]
        ).ravel()
        well.data.loc[:, "observed_TVT_scaled"] = 0.0
        observed_mask = well.data["observed_TVT"].notna()
        well.data.loc[observed_mask, "observed_TVT_scaled"] = (
            well.data.loc[observed_mask, "observed_TVT"] - target_mean
        ) / target_scale
        well.tw_gr_tokens_scaled = (
            (well.tw_gr_tokens - gr_mean) / gr_scale
        ).astype(np.float32)


apply_scalers(train_wells)
apply_scalers(valid_wells)
apply_scalers(test_wells)
print(f"target: mean={target_mean:.1f}, scale={target_scale:.1f}")

Typewell memory: 100%|██████████| 3/3 [00:00<00:00, 1010.11well/s]


target: mean=11502.7, scale=637.7


In [4]:
# Dataset (v1 ile ayni): typewell tokenlari + anchor bilgisi segmentlere eklenir.


class TypewellStatefulTVTWindowDataset(StatefulTVTWindowDataset):
    def __init__(
        self,
        *,
        tw_tvt_grid,
        tw_gr_tokens_scaled,
        target_scale,
        target_mean,
        **kwargs,
    ):
        super().__init__(**kwargs)
        anchor_row = self.prediction_start - 1
        anchor_tvt = float(self.data.loc[anchor_row, "TVT"])
        dtvt = (
            (tw_tvt_grid.astype(np.float64) - anchor_tvt) / target_scale
        ).astype(np.float32)
        self.anchor_tvt = anchor_tvt
        self.anchor_scaled = torch.tensor(
            (anchor_tvt - target_mean) / target_scale, dtype=torch.float32
        )
        self.tw_dtvt = torch.from_numpy(dtvt)
        self.tw_tokens = torch.from_numpy(
            np.concatenate(
                [tw_gr_tokens_scaled, dtvt[:, None]], axis=1
            ).astype(np.float32)
        )

    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        item["tw_tokens"] = self.tw_tokens
        item["tw_dtvt"] = self.tw_dtvt
        item["anchor_scaled"] = self.anchor_scaled
        return item


def make_real_datasets(prepared_wells):
    return {
        well_id: TypewellStatefulTVTWindowDataset(
            data=well.data,
            feature_columns=RNN_FEATURE_COLUMNS,
            target_column="TVT_scaled",
            window_size=WINDOW_SIZE,
            context_length=CONTEXT_LENGTH,
            mode="real",
            prediction_mask_column="is_prediction_row",
            observed_context_column="observed_TVT_scaled",
            tw_tvt_grid=well.tw_tvt_grid,
            tw_gr_tokens_scaled=well.tw_gr_tokens_scaled,
            target_scale=target_scale,
            target_mean=target_mean,
        )
        for well_id, well in prepared_wells.items()
    }


valid_datasets = make_real_datasets(valid_wells)
test_datasets = make_real_datasets(test_wells)

_ds = next(iter(test_datasets.values()))
_seg = _ds[0]
assert _seg["tw_tokens"].shape == (
    TYPEWELL_GRID_POINTS, 2 * TYPEWELL_TOKEN_HALF_WINDOW + 2
)
assert torch.isfinite(_seg["tw_tokens"]).all()
print(f"tw_tokens: {tuple(_seg['tw_tokens'].shape)}, "
      f"features: {tuple(_seg['features'].shape)}")

tw_tokens: (512, 14), features: (256, 22)


In [5]:
class TypewellAttnTVTModelV2(nn.Module):
    """GRU + Conv1d typewell encoder + cift attention (context / pointer).

    v1'den farklar:
    - Typewell tokenlari tek Linear yerine dilation'i artan 3 katmanli
      Conv1d ile encode edilir (cok olcekli GR imzasi).
    - Pointer icin ayri, tek kafali, dropout'suz attention: logit =
      dot(q,k)/sqrt(H) + locality + corr. Softmax'i manuel oldugu icin
      dagilim her modda tam normalize olur.
    - Locality bias: -dtvt^2 / (2*sigma^2), sigma ogrenilebilir. Anchor'a
      yakin derinlikler a priori daha olasi.
    - Corr bias: horizontal GR'in son 13 satiri ile her token pattern'i
      arasindaki NCC, ogrenilebilir gain ile logit'e eklenir.
    - Context okuma v1'deki 4 kafali dropout'lu attention'da kalir.

    Cikti v1 ile ayni pointer-residual formu:
    prediction = anchor_scaled + soft_dtvt + residual.
    """

    def __init__(
        self,
        input_size: int,
        tw_channels: int = 14,
        hidden_size: int = 64,
        num_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.15,
        gr_feature_index: int = GR_FEATURE_INDEX,
        corr_window: int = 2 * TYPEWELL_TOKEN_HALF_WINDOW + 1,
        locality_sigma_init: float = 0.25,
    ):
        super().__init__()
        self.gr_feature_index = gr_feature_index
        self.corr_window = corr_window
        self.hidden_size = hidden_size

        self.rnn_layer = nn.GRU(
            input_size,
            hidden_size,
            n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )

        # (5) Conv1d typewell encoder: kanal boyutu token ozellikleri,
        # uzamsal boyut 512'lik TVT gridi. Dilation 1/2/4 -> ~5/9/17
        # noktalik alici alanlar ust uste biner.
        self.typewell_encoder = nn.Sequential(
            nn.Conv1d(tw_channels, hidden_size, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_size, hidden_size, kernel_size=5,
                      dilation=2, padding=4),
            nn.GELU(),
            nn.Conv1d(hidden_size, hidden_size, kernel_size=5,
                      dilation=4, padding=8),
        )

        # Context yolu (v1 ile ayni ruh): 4 kafali, dropout'lu.
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.context_attn = nn.MultiheadAttention(
            hidden_size, num_heads, batch_first=True, dropout=dropout
        )

        # (4) Pointer yolu: ayri projeksiyonlar, tek kafa, dropout yok.
        self.pointer_q = nn.Linear(hidden_size, hidden_size)
        self.pointer_k = nn.Linear(hidden_size, hidden_size)

        # (3) Ogrenilebilir locality genisligi (scaled TVT birimi).
        self.log_sigma = nn.Parameter(
            torch.tensor(math.log(locality_sigma_init))
        )
        # (6) NCC bias'inin ogrenilebilir kazanci.
        self.corr_gain = nn.Parameter(torch.tensor(1.0))

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size * 2 + 1),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2 + 1, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, 1),
        )

    def _gr_correlation(self, x, tw_tokens):
        """Her timestep'in son `corr_window` GR satiri ile her typewell
        token pattern'i arasinda NCC. Cikti: (B, T, M), [-1, 1]."""
        w = self.corr_window
        gr = x[..., self.gr_feature_index]                    # (B, T)
        padded = F.pad(gr.unsqueeze(1), (w - 1, 0), mode="replicate")
        windows = padded.squeeze(1).unfold(1, w, 1)           # (B, T, w)
        windows = windows - windows.mean(-1, keepdim=True)
        windows = windows / windows.norm(dim=-1, keepdim=True).clamp_min(1e-6)

        patterns = tw_tokens[..., :w]                         # (B, M, w)
        patterns = patterns - patterns.mean(-1, keepdim=True)
        patterns = patterns / patterns.norm(dim=-1, keepdim=True).clamp_min(1e-6)

        return torch.einsum("btw,bmw->btm", windows, patterns)

    def forward(self, x, tw_tokens, tw_dtvt, anchor_scaled, hidden=None):
        # x: (B, T, F), tw_tokens: (B, M, C), tw_dtvt: (B, M),
        # anchor_scaled: (B,)
        rnn_out, hidden = self.rnn_layer(x, hidden)           # (B, T, H)

        memory = self.typewell_encoder(
            tw_tokens.transpose(1, 2)
        ).transpose(1, 2)                                     # (B, M, H)

        # Context yolu. need_weights=True bilerek: False olursa PyTorch
        # hizli SDPA yoluna girer ve MPS backend'i SDPA'da dropout
        # desteklemez (NotImplementedError). Agirliklar kullanilmiyor.
        context, _ = self.context_attn(
            self.q_proj(rnn_out), memory, memory, need_weights=True
        )                                                     # (B, T, H)

        # Pointer yolu: dot-product + locality + corr bias'lari
        logits = torch.einsum(
            "bth,bmh->btm", self.pointer_q(rnn_out), self.pointer_k(memory)
        ) / math.sqrt(self.hidden_size)                       # (B, T, M)

        sigma = self.log_sigma.exp()
        locality = -(tw_dtvt ** 2).unsqueeze(1) / (2.0 * sigma ** 2)
        corr = self._gr_correlation(x, tw_tokens)
        logits = logits + locality + self.corr_gain * corr

        pointer_alpha = torch.softmax(logits, dim=-1)         # (B, T, M)
        soft_dtvt = torch.einsum("btm,bm->bt", pointer_alpha, tw_dtvt)

        head_input = torch.cat(
            [rnn_out, context, soft_dtvt.unsqueeze(-1)], dim=-1
        )
        residual = self.head(head_input).squeeze(-1)          # (B, T)
        predictions = anchor_scaled.unsqueeze(-1) + soft_dtvt + residual
        return predictions, hidden, pointer_alpha, soft_dtvt

In [6]:
model = TypewellAttnTVTModelV2(
    input_size=len(RNN_FEATURE_COLUMNS) + 2,
    tw_channels=2 * TYPEWELL_TOKEN_HALF_WINDOW + 2,
    hidden_size=64,
    num_heads=4,
    n_layers=2,
    dropout=0.15,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-5
)

# --- smoke test ---
_ds = next(iter(test_datasets.values()))
_seg = _ds[0]
_x = _seg["features"].unsqueeze(0).to(device)
_tw = _seg["tw_tokens"].unsqueeze(0).to(device)
_dt = _seg["tw_dtvt"].unsqueeze(0).to(device)
_an = _seg["anchor_scaled"].unsqueeze(0).to(device)

# Pointer softmax'i manuel oldugu icin TRAIN modunda bile tam normalize
# olmali (v1'deki dropout sorunu v2'de yapisal olarak yok).
model.train()
_pred, _hidden, _alpha, _soft = model(_x, _tw, _dt, _an)
assert torch.allclose(
    _alpha.sum(-1), torch.ones_like(_alpha.sum(-1)), atol=1e-5
), "pointer alpha train modunda normalize degil"

model.eval()
with torch.no_grad():
    _pred, _hidden, _alpha, _soft = model(_x, _tw, _dt, _an)

assert _pred.shape == _seg["target"].unsqueeze(0).shape
assert _alpha.shape == (1, _seg["features"].shape[0], TYPEWELL_GRID_POINTS)
assert _soft.shape == _pred.shape
assert torch.isfinite(_pred).all() and torch.isfinite(_alpha).all()

# Locality bias init'te etkili mi: uniform dagilima gore anchor cevresine
# yigilma bekliyoruz (entropi log(M)'den belirgin dusuk olmali).
_entropy = -(_alpha * _alpha.clamp_min(1e-12).log()).sum(-1).mean()
print(f"pointer entropy: {_entropy:.2f} (uniform olsaydi {math.log(TYPEWELL_GRID_POINTS):.2f})")
n_params = sum(p.numel() for p in model.parameters())
print(f"pred: {tuple(_pred.shape)}, alpha: {tuple(_alpha.shape)}")
print(f"parameters: {n_params:,} (v1: 72,259)")

pointer entropy: 5.80 (uniform olsaydi 6.24)
pred: (1, 256), alpha: (1, 256, 512)
parameters: 125,253 (v1: 72,259)


In [7]:
# Egitim/degerlendirme fonksiyonlari. v1'den tek fark run_batch'te:
# (1) pointer loss: soft_dtvt -> (target - anchor_scaled) MSE,
# (2) smoothness: ardisik soft_dtvt farklarinin karesi (her iki satiri da
#     loss bolgesinde olan ciftlerde).


def detach_hidden(hidden):
    if isinstance(hidden, tuple):
        return tuple(state.detach() for state in hidden)
    return hidden.detach()


def masked_mse_loss(predictions, targets, loss_mask):
    return ((predictions - targets) ** 2)[loss_mask].mean()


def sample_prediction_start(well, rng):
    num_rows = len(well.data)
    min_forecast = max(
        MIN_FORECAST_LENGTH,
        int(num_rows * MIN_FORECAST_FRACTION),
    )
    first = CONTEXT_LENGTH
    last = num_rows - min_forecast
    if last < first:
        return None
    if rng.random() < REAL_PS_ANCHOR_PROB:
        low = max(first, int(num_rows * REAL_PS_FRACTION_RANGE[0]))
        high = min(last, int(num_rows * REAL_PS_FRACTION_RANGE[1]))
        if low <= high:
            return rng.randint(low, high)
    return rng.randint(first, last)


def make_batch_segments(max_prefix, total_len):
    segments = []
    for start in range(0, max_prefix, WINDOW_SIZE):
        segments.append((start, min(start + WINDOW_SIZE, max_prefix)))
    first_end = min(total_len, max_prefix + CONTEXT_LENGTH + WINDOW_SIZE)
    segments.append((max_prefix, first_end))
    for start in range(first_end, total_len, WINDOW_SIZE):
        segments.append((start, min(start + WINDOW_SIZE, total_len)))
    return segments


def build_episode_batch(episode_datasets):
    prefix_lens = [
        max(0, ds.prediction_start - CONTEXT_LENGTH)
        for ds in episode_datasets
    ]
    main_lens = [
        len(ds.features) - prefix
        for ds, prefix in zip(episode_datasets, prefix_lens)
    ]
    max_prefix = max(prefix_lens)
    max_main = max(main_lens)
    total_len = max_prefix + max_main

    batch_size = len(episode_datasets)
    feature_dim = episode_datasets[0].features.shape[1]
    features = torch.zeros(batch_size, total_len, feature_dim)
    targets = torch.zeros(batch_size, total_len)
    loss_mask = torch.zeros(batch_size, total_len, dtype=torch.bool)

    for i, (ds, prefix) in enumerate(zip(episode_datasets, prefix_lens)):
        offset = max_prefix - prefix
        rows = len(ds.features)
        features[i, offset : offset + rows] = ds.features
        targets[i, offset : offset + rows] = ds.target
        loss_mask[i, offset : offset + rows] = ds.loss_mask

    return {
        "features": features,
        "targets": targets,
        "loss_mask": loss_mask,
        "tw_tokens": torch.stack([ds.tw_tokens for ds in episode_datasets]),
        "tw_dtvt": torch.stack([ds.tw_dtvt for ds in episode_datasets]),
        "anchor_scaled": torch.stack(
            [ds.anchor_scaled for ds in episode_datasets]
        ),
        "segments": make_batch_segments(max_prefix, total_len),
    }


def run_batch(model, batch, optimizer=None):
    features = batch["features"].to(device)
    targets = batch["targets"].to(device)
    loss_mask = batch["loss_mask"].to(device)
    tw_tokens = batch["tw_tokens"].to(device)
    tw_dtvt = batch["tw_dtvt"].to(device)
    anchor_scaled = batch["anchor_scaled"].to(device)

    squared_error_sum = 0.0
    observation_count = 0
    hidden = None

    for start, end in batch["segments"]:
        x = features[:, start:end]
        y = targets[:, start:end]
        mask = loss_mask[:, start:end]

        if optimizer is not None and not mask.any():
            with torch.no_grad():
                _, hidden, _, _ = model(
                    x, tw_tokens, tw_dtvt, anchor_scaled, hidden
                )
            hidden = detach_hidden(hidden)
            continue

        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
            predictions, hidden, _, soft_dtvt = model(
                x, tw_tokens, tw_dtvt, anchor_scaled, hidden
            )
            task_loss = masked_mse_loss(predictions, y, mask)

            # (1) Pointer'in gercek hedefi: dtvt = target - anchor (ikisi
            # de scaled TVT biriminde oldugu icin fark dogrudan dtvt'dir).
            true_dtvt = y - anchor_scaled.unsqueeze(-1)
            pointer_loss = masked_mse_loss(soft_dtvt, true_dtvt, mask)

            # (2) Smoothness: her iki ucu da loss bolgesindeki ciftlerde.
            pair_mask = mask[:, 1:] & mask[:, :-1]
            if pair_mask.any():
                jumps = (soft_dtvt[:, 1:] - soft_dtvt[:, :-1])[pair_mask]
                smooth_loss = (jumps ** 2).mean()
            else:
                smooth_loss = soft_dtvt.new_zeros(())

            loss = (
                task_loss
                + POINTER_LOSS_WEIGHT * pointer_loss
                + SMOOTHNESS_LOSS_WEIGHT * smooth_loss
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            predictions = predictions.detach()
        else:
            predictions, hidden, _, _ = model(
                x, tw_tokens, tw_dtvt, anchor_scaled, hidden
            )
        hidden = detach_hidden(hidden)

        if mask.any():
            errors_tvt = (predictions - y) * target_scale
            squared_error_sum += torch.sum(errors_tvt[mask] ** 2).item()
            observation_count += int(mask.sum().item())

    return squared_error_sum, observation_count


def make_episode_dataset(well, prediction_start):
    known = well.data.iloc[:prediction_start]
    known_tvt, known_gr = known_calibration_pairs(
        known, "TVT", gr_column="GR_raw"
    )
    scale, offset = fit_typewell_gr_calibration(
        known_tvt, known_gr, well.tw_tvt_grid, well.tw_gr_grid
    )
    tokens_scaled = (
        (well.tw_gr_tokens_raw * scale + offset - gr_mean) / gr_scale
    ).astype(np.float32)

    return TypewellStatefulTVTWindowDataset(
        data=well.data,
        feature_columns=RNN_FEATURE_COLUMNS,
        target_column="TVT_scaled",
        window_size=WINDOW_SIZE,
        context_length=CONTEXT_LENGTH,
        mode="synthetic",
        prediction_start=prediction_start,
        tw_tvt_grid=well.tw_tvt_grid,
        tw_gr_tokens_scaled=tokens_scaled,
        target_scale=target_scale,
        target_mean=target_mean,
    )


def train_one_epoch(model, prepared_wells, optimizer, rng, epoch=None):
    model.train()

    episodes = []
    for well_id, well in prepared_wells.items():
        for _ in range(ANCHORS_PER_WELL):
            prediction_start = sample_prediction_start(well, rng)
            if prediction_start is not None:
                episodes.append((well_id, prediction_start))
    episodes.sort(key=lambda item: len(prepared_wells[item[0]].data))

    bucket_size = EPISODE_BATCH_SIZE * 4
    for i in range(0, len(episodes), bucket_size):
        bucket = episodes[i : i + bucket_size]
        rng.shuffle(bucket)
        episodes[i : i + bucket_size] = bucket

    batches = [
        episodes[i : i + EPISODE_BATCH_SIZE]
        for i in range(0, len(episodes), EPISODE_BATCH_SIZE)
    ]
    rng.shuffle(batches)

    squared_error_sum = 0.0
    observation_count = 0
    progress = tqdm(
        batches,
        desc=f"Epoch {epoch}" if epoch is not None else "Train",
        unit="batch",
        leave=False,
    )
    for batch_episodes in progress:
        episode_datasets = [
            make_episode_dataset(prepared_wells[well_id], prediction_start)
            for well_id, prediction_start in batch_episodes
        ]
        batch = build_episode_batch(episode_datasets)
        sq_err, n_obs = run_batch(model, batch, optimizer)
        squared_error_sum += sq_err
        observation_count += n_obs

        if observation_count > 0:
            progress.set_postfix(
                running_rmse=f"{np.sqrt(squared_error_sum / observation_count):.4f}"
            )

    if observation_count == 0:
        raise ValueError("Training epoch'unda loss satiri bulunamadi.")
    return float(np.sqrt(squared_error_sum / observation_count))


@torch.no_grad()
def evaluate_real_ps(model, datasets, batch_size=EPISODE_BATCH_SIZE):
    model.eval()

    ordered = sorted(datasets.values(), key=lambda ds: len(ds.features))
    batches = [
        ordered[i : i + batch_size]
        for i in range(0, len(ordered), batch_size)
    ]

    squared_error_sum = 0.0
    observation_count = 0
    for batch_datasets in tqdm(
        batches, desc="Eval", unit="batch", leave=False
    ):
        batch = build_episode_batch(batch_datasets)
        sq_err, n_obs = run_batch(model, batch, optimizer=None)
        squared_error_sum += sq_err
        observation_count += n_obs

    if observation_count == 0:
        raise ValueError("Gercek PS sonrasinda skorlanacak satir yok.")
    return float(np.sqrt(squared_error_sum / observation_count))

In [ ]:
history = []
best_valid_rmse = float("inf")
best_state = None
anchor_rng = random.Random(SEED)

for epoch in range(1, NUM_EPOCHS + 1):
    train_rmse = train_one_epoch(
        model, train_wells, optimizer, anchor_rng, epoch=epoch
    )
    valid_rmse = evaluate_real_ps(model, valid_datasets)
    history.append({
        "epoch": epoch,
        "synthetic_train_rmse": train_rmse,
        "real_ps_valid_rmse": valid_rmse,
    })
    if valid_rmse < best_valid_rmse:
        best_valid_rmse = valid_rmse
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"Epoch {epoch:02d} | synthetic train RMSE: {train_rmse:.4f} | "
        f"real-PS valid RMSE: {valid_rmse:.4f} | "
        f"sigma={model.log_sigma.exp().item():.3f} | "
        f"corr_gain={model.corr_gain.item():.3f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), CHECKPOINT_PATH)
# Final raporlama kuyu kuyu (batch_size=1): pad kaynakli hicbir etki yok.
test_rmse = evaluate_real_ps(model, test_datasets, batch_size=1)
print(f"Typewell-attention v2 reserved test RMSE: {test_rmse:.4f}")
pd.DataFrame(history)

Epoch 01 | synthetic train RMSE: 38.7512 | real-PS valid RMSE: 20.6162 | sigma=0.243 | corr_gain=-0.009


Epoch 02 | synthetic train RMSE: 30.5104 | real-PS valid RMSE: 21.3478 | sigma=0.235 | corr_gain=-0.000


Epoch 3:  65%|██████▍   | 106/164 [08:27<04:42,  4.87s/batch, running_rmse=30.7164]

In [ ]:
import matplotlib.pyplot as plt

# Pointer attention haritasi: v2'de artik pointer'a ozel dagilim cizilir.
# Beklenen goruntu: gercek TVT cizgisini takip eden koyu bir serit.
# v1'deki uniform haritayla dogrudan karsilastirilabilir.


@torch.no_grad()
def collect_attention_map(model, dataset):
    model.eval()
    hidden = None
    tw_tokens = dataset.tw_tokens.unsqueeze(0).to(device)
    tw_dtvt = dataset.tw_dtvt.unsqueeze(0).to(device)
    anchor_scaled = dataset.anchor_scaled.unsqueeze(0).to(device)

    alphas, preds, softs = [], [], []
    for segment in dataset:
        features = segment["features"].unsqueeze(0).to(device)
        predictions, hidden, alpha, soft_dtvt = model(
            features, tw_tokens, tw_dtvt, anchor_scaled, hidden
        )
        alphas.append(alpha.squeeze(0).cpu())
        preds.append(predictions.squeeze(0).cpu())
        softs.append(soft_dtvt.squeeze(0).cpu())
    return torch.cat(alphas), torch.cat(preds), torch.cat(softs)


_well_id = max(
    test_datasets,
    key=lambda wid: int(test_wells[wid].data["is_prediction_row"].sum()),
)
_ds = test_datasets[_well_id]
_alpha_map, _preds, _softs = collect_attention_map(model, _ds)

_well = test_wells[_well_id]
_true_tvt = _well.data["TVT"].to_numpy()
_pred_tvt = _preds.numpy() * target_scale + target_mean
_pointer_tvt = _softs.numpy() * target_scale + _ds.anchor_tvt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].imshow(
    _alpha_map.numpy().T,
    aspect="auto",
    origin="lower",
    extent=[0, len(_alpha_map), _well.tw_tvt_grid[0], _well.tw_tvt_grid[-1]],
    cmap="viridis",
)
axes[0].plot(np.arange(len(_true_tvt)), _true_tvt, "r-", lw=0.8, label="true TVT")
axes[0].axvline(_ds.prediction_start, color="w", ls="--", lw=1, label="PS")
axes[0].set_ylabel("typewell TVT")
axes[0].set_title(f"Pointer attention map (v2) — {_well_id}")
axes[0].legend(loc="upper right")

axes[1].plot(_true_tvt, label="true TVT", lw=1)
axes[1].plot(_pred_tvt, label="predicted TVT", lw=1)
axes[1].plot(_pointer_tvt, label="pointer (anchor + soft_dtvt)", lw=0.8, alpha=0.7)
axes[1].axvline(_ds.prediction_start, color="k", ls="--", lw=1, label="PS")
axes[1].set_xlabel("row (MD sirasi)")
axes[1].set_ylabel("TVT")
axes[1].legend()

plt.tight_layout()
plt.show()

_pred_region = slice(_ds.prediction_start, len(_true_tvt))
_rmse = float(np.sqrt(np.mean(
    (_pred_tvt[_pred_region] - _true_tvt[_pred_region]) ** 2
)))
_pointer_rmse = float(np.sqrt(np.mean(
    (_pointer_tvt[_pred_region] - _true_tvt[_pred_region]) ** 2
)))
print(f"{_well_id} prediction-region RMSE: {_rmse:.3f} "
      f"(pointer tek basina: {_pointer_rmse:.3f})")